# 🔥 Advanced · DPO — preference alignment

> 🔥 **Advanced / heavy lab.** Teach an LLM which answers people prefer, using chosen/rejected pairs (DPO — the RLHF-free recipe).
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **20–40 min** including downloads. Built on the official **[TRL DPOTrainer](https://huggingface.co/docs/trl/dpo_trainer)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

Run this *after* SFT (the QLoRA lab) — alignment is the second stage of the modern LLM pipeline.

In [ ]:
!nvidia-smi -L
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes

## 1 · Load the (SFT) model in 4-bit

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
base = "Qwen/Qwen2.5-0.5B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, quantization_config=bnb, device_map="auto")

## 2 · Preference data — prompt + chosen + rejected

In [ ]:
from datasets import load_dataset
ds = load_dataset("trl-lib/ultrafeedback_binarized", split="train[:2000]")
print(ds.column_names)

## 3 · Train with DPO (LoRA adapters, no separate reward model)

In [ ]:
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig
args = DPOConfig(output_dir="dpo-out", per_device_train_batch_size=2, gradient_accumulation_steps=4,
                 max_steps=200, learning_rate=5e-6, logging_steps=20, bf16=True, beta=0.1, report_to="none")
trainer = DPOTrainer(model=model, args=args, train_dataset=ds, processing_class=tok,
                     peft_config=LoraConfig(r=16, lora_alpha=32, task_type="CAUSAL_LM", target_modules="all-linear"))
trainer.train()

## 4 · Inspect
Watch `rewards/chosen` rise above `rewards/rejected` and the `rewards/accuracies` climb in the logs — that's the model learning the preference. Save:

In [ ]:
trainer.save_model("dpo-adapter")

## Notes & next steps
- DPO needs an SFT'd base; using a raw base model gives weak results.
- Variants: **IPO**, **KTO**, **ORPO** (one-stage) — all in TRL.
- Lower `beta` = trust the data more; raise it = stay closer to the reference model.